# AgentWeb Integration with FinGPT

This notebook demonstrates how to use AgentWeb's business data API with FinGPT agents for enhanced financial analysis and compliance verification.

## What is AgentWeb?

AgentWeb provides access to 85M+ businesses across 195 countries with:
- Business search and verification
- Local search capabilities
- Contact information retrieval
- Real-time data updates

## Setup

First, get your free API key from https://agentweb.live (1,000 requests/day free)

In [ ]:
# Install dependencies if needed
!pip install httpx -q

In [ ]:
# Set your AgentWeb API key
import os
os.environ['AGENTWEB_API_KEY'] = 'your_agentweb_api_key_here'  # Replace with your actual key

## Basic AgentWeb Client Usage

Let's start with the basic AgentWeb client to search for businesses.

In [ ]:
import asyncio
import sys
sys.path.append('/Users/test/FinGPT')

from finogrid.fingpt_integration.agentweb import AgentWebClient

# Initialize the client
client = AgentWebClient()

# Check if configured
print(f"AgentWeb configured: {client.is_configured()}")

### Search for Businesses

In [ ]:
async def search_business_example():
    """Example: Search for financial services businesses."""
    result = await client.search_businesses(
        query="financial services",
        country="US",
        limit=5
    )
    return result

# Run the search
result = asyncio.run(search_business_example())
print(result)

### Verify Business Entity

In [ ]:
async def verify_business_example():
    """Example: Verify a specific business entity."""
    result = await client.verify_business_entity(
        business_name="JPMorgan Chase",
        country="US"
    )
    return result

# Run verification
verification = asyncio.run(verify_business_example())
print(f"Verification result: {verification}")

### Local Search

In [ ]:
async def local_search_example():
    """Example: Local search for businesses in a specific area."""
    result = await client.local_search(
        query="coffee shops",
        location="Manhattan, New York",
        limit=3
    )
    return result

# Run local search
local_results = asyncio.run(local_search_example())
print(local_results)

## Integration with FinGPT Agents

Now let's see how AgentWeb integrates with FinGPT's agents for enhanced capabilities.

### AuditGovernanceAgent with Business Verification

The AuditGovernanceAgent can now verify business entities for compliance checks.

In [ ]:
import sys
sys.path.append('/Users/test/FinGPT/finogrid')

from agents.audit_governance.agent import AuditGovernanceAgent

# Initialize agent with AgentWeb client
audit_agent = AuditGovernanceAgent(
    db_session_factory=None,  # Would be database session in production
    knowledge_base=None,       # Would be RAG knowledge base in production
    llm_client=None,           # Would be LLM client in production
    agentweb_client=client    # AgentWeb client for business verification
)

print("AuditGovernanceAgent initialized with AgentWeb support")

In [ ]:
async def verify_with_audit_agent():
    """Use AuditGovernanceAgent to verify a business."""
    result = await audit_agent.verify_business_entity(
        business_name="Bank of America",
        country="US"
    )
    return result

# Run verification through agent
agent_verification = asyncio.run(verify_with_audit_agent())
print(f"Agent verification result: {agent_verification}")

### TreasuryStrategyAgent with Business Intelligence

The TreasuryStrategyAgent can gather business intelligence for corridor analysis.

In [ ]:
from agents.treasury_strategy.agent import TreasuryStrategyAgent

# Initialize agent with AgentWeb client
treasury_agent = TreasuryStrategyAgent(
    corridor_forecaster=None,    # Would be forecaster in production
    db_session_factory=None,     # Would be database session in production
    agentweb_client=client      # AgentWeb client for business intelligence
)

print("TreasuryStrategyAgent initialized with AgentWeb support")

In [ ]:
async def get_corridor_intelligence():
    """Get business intelligence for a specific corridor."""
    result = await treasury_agent.get_corridor_business_intelligence(
        corridor="NG",  # Nigeria corridor
        search_query="financial services"
    )
    return result

# Get corridor intelligence
intelligence = asyncio.run(get_corridor_intelligence())
print(f"Corridor intelligence: {intelligence}")

In [ ]:
async def enhance_corridor_analysis():
    """Enhance corridor analysis with business context."""
    context = await treasury_agent.enhance_corridor_analysis(
        corridor="BR",  # Brazil corridor
        business_context="banking"
    )
    return context

# Get enhanced analysis
enhanced_context = asyncio.run(enhance_corridor_analysis())
print(f"Enhanced corridor context: {enhanced_context}")

## Practical Use Cases

### 1. KYB (Know Your Business) Verification
Use AgentWeb to verify business entities during onboarding:

In [ ]:
async def kyb_verification():
    """Comprehensive KYB verification example."""
    business_name = "Acme Corporation"
    country = "US"
    
    # Step 1: Verify business exists
    verification = await client.verify_business_entity(business_name, country)
    
    if verification['verified']:
        print(f"✓ Business verified: {business_name}")
        
        # Step 2: Get detailed information
        business_data = verification['business_data']
        print(f"  Category: {business_data.get('category')}")
        print(f"  Address: {business_data.get('address')}")
        print(f"  Phone: {business_data.get('phone')}")
        print(f"  Rating: {business_data.get('rating')}")
        
        # Step 3: Get contact information
        if business_data.get('id'):
            contacts = await client.get_business_contacts(business_data['id'])
            print(f"  Website: {contacts.get('website')}")
            print(f"  Email: {contacts.get('email')}")
    else:
        print(f"✗ Verification failed: {verification['message']}")

# Run KYB verification
asyncio.run(kyb_verification())

### 2. Corridor Market Analysis
Analyze business landscape for different payout corridors:

In [ ]:
async def analyze_corridor_markets():
    """Analyze business landscape across multiple corridors."""
    corridors = ['NG', 'BR', 'IN', 'PH']  # Nigeria, Brazil, India, Philippines
    
    for corridor in corridors:
        print(f"\n=== Corridor: {corridor} ===")
        intelligence = await treasury_agent.get_corridor_business_intelligence(
            corridor=corridor,
            search_query="financial services"
        )
        
        if 'error' not in intelligence:
            print(f"Country: {intelligence['country']}")
            print(f"Businesses found: {intelligence['total_businesses_found']}")
            print(f"Top categories: {intelligence['category_distribution']}")
        else:
            print(f"Error: {intelligence['error']}")

# Run corridor analysis
asyncio.run(analyze_corridor_markets())

### 3. Compliance Context Enhancement
Enhance compliance reports with business context:

In [ ]:
async def enhance_compliance_report():
    """Enhance compliance report with business context."""
    entities = ["Citibank", "HSBC", "Standard Chartered"]
    
    print("=== Compliance Report Enhancement ===")
    for entity in entities:
        context = await audit_agent.enhance_compliance_context(entity, "US")
        print(f"\n{entity}:")
        print(f"  {context}")

# Run compliance enhancement
asyncio.run(enhance_compliance_report())

## Error Handling and Fallbacks

The integration includes proper error handling for when AgentWeb is not configured:

In [ ]:
# Test with unconfigured client
unconfigured_client = AgentWebClient(api_key="")
print(f"Configured: {unconfigured_client.is_configured()}")

async def test_error_handling():
    result = await unconfigured_client.search_businesses("test", "US")
    print(f"Error handling result: {result}")

asyncio.run(test_error_handling())

## Summary

This notebook demonstrated:

1. **Basic AgentWeb Usage**: Business search, verification, and local search
2. **Agent Integration**: How AuditGovernanceAgent and TreasuryStrategyAgent use AgentWeb
3. **Practical Use Cases**: KYB verification, corridor analysis, compliance enhancement
4. **Error Handling**: Graceful fallbacks when AgentWeb is not configured

## Next Steps

- Get your free API key from https://agentweb.live
- Integrate with your database and LLM clients for full functionality
- Add AgentWeb to your agent initialization in production
- Monitor usage (free tier: 1,000 requests/day)

## References

- AgentWeb: https://agentweb.live
- AgentWeb API Docs: https://api.agentweb.live/docs
- FinGPT Documentation: https://github.com/AI4Finance-Foundation/FinGPT